## Google Colab Setup (Google Drive)
Run this cell to mount your Google Drive and set the working directory to your project folder.


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    # Change directory to where your dataset is stored
    os.chdir('/content/drive/MyDrive/DL_mask_detection_project')
    print("\nCurrent Working Directory set to:", os.getcwd())
except ImportError:
    print("Not running in Google Colab. Skipping Drive mount.")


In [ ]:
import os
import cv2
import numpy as np
import xml.etree.ElementTree as ET
from tqdm import tqdm
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dropout, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 10
LABEL_MAP = {"with_mask": 0, "without_mask": 1, "mask_weared_incorrect": 2}


In [ ]:
# Data Extraction Cell
X = []
y = []

images_dir = os.path.join(os.getcwd(), "demo_images")
annotations_dir = os.path.join(os.getcwd(), "annotations")

if not os.path.exists(images_dir) or not os.path.exists(annotations_dir):
    print("Warning: demo_images or annotations directory not found in:", os.getcwd())
else:
    xml_files = [f for f in os.listdir(annotations_dir) if f.endswith('.xml')]
    if len(xml_files) == 0:
        print("Warning: No XML files found in annotations directory.")
    else:
        for xml_file in tqdm(xml_files, desc="Processing xml files"):
            tree = ET.parse(os.path.join(annotations_dir, xml_file))
            root = tree.getroot()
            img_filename = root.find("filename").text
            img_path = os.path.join(images_dir, img_filename)
            
            if not os.path.exists(img_path):
                continue
                
            img = cv2.imread(img_path)
            if img is None:
                continue
                
            for obj in root.findall("object"):
                label = obj.find("name").text
                if label not in LABEL_MAP:
                    continue
                    
                bndbox = obj.find("bndbox")
                xmin = int(bndbox.find("xmin").text)
                ymin = int(bndbox.find("ymin").text)
                xmax = int(bndbox.find("xmax").text)
                ymax = int(bndbox.find("ymax").text)
                
                face = img[ymin:ymax, xmin:xmax]
                if face.size == 0:
                    continue
                    
                face = cv2.resize(face, (IMG_SIZE, IMG_SIZE))
                face = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)
                face = face / 255.0
                
                X.append(face)
                y.append(LABEL_MAP[label])

X = np.array(X, dtype=np.float32)
y = np.array(y)


In [ ]:
# Preprocessing Cell
if len(y) > 0:
    y = to_categorical(y, num_classes=3)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    datagen = ImageDataGenerator()
    print(f"Training size: {len(X_train)} | Testing size: {len(X_test)}")
else:
    print("No valid faces extracted. Preprocessing skipped.")


In [ ]:
# Model Cell
if len(y) > 0:
    base_model = MobileNetV2(weights="imagenet", include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
    for layer in base_model.layers:
        layer.trainable = False
        
    x_tensor = base_model.output
    x_tensor = GlobalAveragePooling2D()(x_tensor)
    x_tensor = Dropout(0.5)(x_tensor)
    predictions = Dense(3, activation="softmax")(x_tensor)
    
    model = Model(inputs=base_model.input, outputs=predictions)
    model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
    model.summary()


In [ ]:
# Training Cell
if len(y) > 0:
    history = model.fit(
        datagen.flow(X_train, y_train, batch_size=BATCH_SIZE),
        validation_data=(X_test, y_test),
        epochs=EPOCHS
    )
    
    model.save(os.path.join(os.getcwd(), "mobilenetv2_mask.h5"))
    print("Model saved to mobilenetv2_mask.h5 in current directory.")


In [ ]:
# Visualization Cell
if len(y) > 0:
    plt.figure(figsize=(10, 5))
    plt.plot(history.history["accuracy"], label="train_acc")
    plt.plot(history.history["val_accuracy"], label="val_acc")
    plt.plot(history.history["loss"], label="train_loss")
    plt.plot(history.history["val_loss"], label="val_loss")
    plt.title("Training Loss and Accuracy")
    plt.xlabel("Epochs")
    plt.ylabel("Loss/Accuracy")
    plt.legend(loc="lower left")
    plt.show()
